# Lab-0. Connect to spark claster with Delta Lake support

In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = SparkSession.builder \
    .appName("FabricSimulation") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.warehouse.dir", "lab_0/lakehouse") # Важливо для імітації таблиць Fabric

spark = configure_spark_with_delta_pip(builder).getOrCreate()
print(f"Spark version: {spark.version} with Delta support")

Spark version: 3.5.0 with Delta support


## Run this cells to check Delta lake support 

In [5]:
df = spark.createDataFrame([(1, "Initial Data")], ["id", "content"])
try:
    df.write.format("delta").mode("overwrite").saveAsTable("my_bronze_table")
    print(f"Data stored in : tmy_bronze_table")
except Exception as e:
    print(f"Error while Save: {e}")


Data stored in : tmy_bronze_table


In [6]:
from delta.tables import DeltaTable
deltaTable = DeltaTable.forPath(spark, "lab_0/lakehouse/my_bronze_table")
history_df = deltaTable.history().select("version", "timestamp", "operation")
history_df.show()

+-------+--------------------+--------------------+
|version|           timestamp|           operation|
+-------+--------------------+--------------------+
|      1|2026-02-01 10:33:...|CREATE OR REPLACE...|
|      0|2026-02-01 10:25:...|CREATE OR REPLACE...|
+-------+--------------------+--------------------+



In [7]:
spark.sql("SELECT * FROM my_bronze_table").show()

+---+------------+
| id|     content|
+---+------------+
|  1|Initial Data|
+---+------------+

